# Falcon prompt/weight separation matrix

This bounded Kaggle GPU experiment is run only after prompt selection on the separate development/validation batteries. It evaluates stock/default, stock/selected, tiny-overfit-adapter/default, and tiny-overfit-adapter/selected on the untouched frozen 24-case battery. It is diagnostic only: no checkpoint is promoted or persisted.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
OUT = WORK / 'falcon-prompt-weight-matrix'
BRANCH = 'research/edge35-adaptive-streaming'
MODEL = 'tiiuae/Falcon-H1-1.5B-Deep-Instruct'
REVISION = 'b6648636ddc906688974282de6e7a243395f5423'
OUT.mkdir(parents=True, exist_ok=True)
def run(command, cwd=REPO, name='command.log'):
    log = OUT / name
    log.parent.mkdir(parents=True, exist_ok=True)
    print('STREAM', ' '.join(map(str, command)), flush=True)
    with log.open('a', encoding='utf-8', buffering=1) as handle:
        proc = subprocess.Popen([str(x) for x in command], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
            rendered = f'[{stamp}] {line}'
            print(rendered, end='', flush=True); handle.write(rendered); handle.flush()
        code = proc.wait()
        print(f'[{time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}] EXIT={code}', flush=True)
        if code: raise RuntimeError(f'command failed: {command}')
if REPO.exists(): shutil.rmtree(REPO)
run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/qeinstein/adtc-llm-limited-hardware.git',str(REPO)], cwd=WORK, name='clone.log')
run([sys.executable,'-m','pip','install','-q','-r','requirements-falcon-production.txt'], name='pip-base.log')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True, capture_output=True, check=False).stdout.strip()
if 'P100' in gpu:
    run([sys.executable,'-m','pip','install','-q','--upgrade','numpy<2'], name='numpy-p100.log')
    run([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','torch==2.6.0','--index-url','https://download.pytorch.org/whl/cu118'], name='torch-p100.log')
    run([sys.executable,'-m','pip','uninstall','-y','torchao','torchvision','torchaudio','bitsandbytes'], name='optional-uninstall.log')
    run([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','--no-deps','transformers==4.53.3','tokenizers==0.21.4','peft==0.15.2','accelerate==1.7.0'], name='hf-stack-final.log')
print(json.dumps({'gpu':gpu,'repo_sha':subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,text=True,capture_output=True,check=True).stdout.strip()}, indent=2), flush=True)


In [ ]:
# The selected file is produced only after prompt development/validation.
selected_path = REPO / 'docs/research/falcon_system_prompt_selected.json'
if not selected_path.is_file(): raise RuntimeError('No validation-selected prompt; do not run the matrix yet')
selected = json.loads(selected_path.read_text())
if selected.get('status') != 'SELECTED_FOR_FROZEN_MATRIX_ONLY': raise RuntimeError(f"selected prompt is not eligible for matrix: {selected.get('status')}")
config = json.loads((REPO / 'configs/falcon-production-v1.json').read_text())
matrix = {'schema_version':'1.0.0','experiment_id':'falcon-frozen-prompt-weight-matrix','candidates':[{'id':'SYS-DEFAULT','family':'default','text':config['data']['system_prompt']}, {'id':selected['id'],'family':selected.get('family','selected'),'text':selected['text']}] }
matrix_path = OUT / 'matrix_candidates.json'
matrix_path.write_text(json.dumps(matrix, indent=2, ensure_ascii=False)+'\n')
battery = REPO / 'docs/research/falcon_probe_heldout.json'
if not battery.is_file(): raise RuntimeError('frozen battery missing')
proof = OUT / 'tiny-overfit-proof'
run([sys.executable,'-u','scripts/falcon_trainability_probe.py','--model',MODEL,'--revision',REVISION,'--output-dir',str(proof),'--steps','64','--learning-rate','0.001'], name='tiny-overfit-proof.log')
adapter = proof / 'adapter'
manifest = json.loads((proof / 'trainability_manifest.json').read_text())
if manifest.get('status') != 'pass' or not adapter.is_dir(): raise RuntimeError('trainability proof did not produce a valid adapter')
for variant, adapter_arg in [('stock', None), ('tiny_overfit_adapter', adapter)]:
    variant_out = OUT / variant
    command = [sys.executable,'-u','scripts/evaluate_falcon_frozen_matrix.py','--model',MODEL,'--revision',REVISION,'--candidates',str(matrix_path),'--battery',str(battery),'--out',str(variant_out)]
    if adapter_arg is not None: command += ['--adapter', str(adapter_arg)]
    run(command, name=f'{variant}.log')
summary = {'schema_version':'1.0.0','experiment_id':'falcon-frozen-prompt-weight-matrix','model':MODEL,'revision':REVISION,'gpu':gpu,'matrix':matrix,'frozen_battery':str(battery),'tiny_overfit_manifest':manifest,'results':{}}
for variant in ('stock','tiny_overfit_adapter'):
    summary['results'][variant] = json.loads((OUT / variant / 'frozen_matrix.json').read_text())
(OUT / 'matrix_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False)+'\n')
print('FROZEN_MATRIX_COMPLETE', json.dumps({'variants':list(summary['results']), 'battery_count':24}, indent=2), flush=True)
